In [1]:
# ============================================================
# Spectral learning under noise: coefficient degradation and
# function reconstruction examples
#
# For a 1D target function (x^2, Fourier basis) and a 2D target
# function (sin(x1^2 * exp(x2)), Legendre basis), this script:
#
#   - Fits sparse spectral regression (Lasso) on centered-whitened
#     design matrices across a range of noise levels sigma/sigma*
#   - Tracks the learned spectral coefficients, their distributions,
#     and the normalized spectral entropy of the coefficient power
#     spectrum as noise increases
#   - Reconstructs and plots the learned functions at three
#     representative noise levels (sigma/sigma* = 0, 1, 10)
#
# Whitening is applied consistently: the design matrix is centered
# first, then whitened via (Gc + rho I)^{-1/2}; raw coefficients
# and the intercept are recovered by back-transformation for
# plotting in the original basis coordinates.
#
# Output: two combined multi-panel PDF figures (1D and 2D),
#         saved to the specified output directory.
#
# Dependencies: Random, LinearAlgebra, Statistics, Printf,
#               Distributions, PyPlot, Lasso, Serialization,
#               LaTeXStrings, ProgressMeter
# ============================================================

using Random, LinearAlgebra, Statistics, Printf
using Distributions
using PyPlot
using Lasso
using Serialization
using LaTeXStrings
using ProgressMeter

# ------------------------------------------------------------
# Global style variables (applied per-figure/per-axes)
# ------------------------------------------------------------
const FONT_TYPE        = "Arial"
const FONT_SIZE        = 18
const LABEL_SIZE       = 18
const TITLE_SIZE       = 16
const TICK_LABEL_SIZE  = 16

const AXES_LINEWIDTH   = 1.8
const TICK_MAJOR_WIDTH = 1.1
const TICK_MINOR_WIDTH = 0.8
const TICK_MAJOR_SIZE  = 4.5
const TICK_MINOR_SIZE  = 2.5

# ------------------------------------------------------------
# Combined figure layout
# ------------------------------------------------------------
const FIG_L   = 0.08
const FIG_R   = 0.02
const FIG_T   = 0.03
const FIG_B   = 0.055
const FIG_HG  = 0.055
const FIG_VG  = 0.055

const ROW_H   = 0.18
const ROW4_H  = 0.18

# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------
mse(y, yhat)  = mean((y .- yhat).^2)
rmse(y, yhat) = sqrt(mse(y, yhat))
ensure_dir(path::String) = (isdir(path) || mkpath(path); path)

function save_pdf(path_noext::String; dpi::Int=300, bbox_inches=nothing, pad_inches=nothing)
    kwargs = Dict{Symbol,Any}(:dpi => dpi)
    if bbox_inches !== nothing
        kwargs[:bbox_inches] = bbox_inches
    end
    if pad_inches !== nothing
        kwargs[:pad_inches] = pad_inches
    end
    savefig(path_noext * ".pdf"; kwargs...)
end

# ------------------------------------------------------------
# Font / styling helpers
# ------------------------------------------------------------
function set_font!(txt; size::Real=FONT_SIZE)
    txt.set_fontname(FONT_TYPE)
    txt.set_fontsize(size)
end

function style_tick_fonts!(ax; is3d::Bool=false)
    for lab in ax.get_xticklabels()
        set_font!(lab; size=TICK_LABEL_SIZE)
    end
    for lab in ax.get_yticklabels()
        set_font!(lab; size=TICK_LABEL_SIZE)
    end
    if is3d
        for lab in ax.get_zticklabels()
            set_font!(lab; size=TICK_LABEL_SIZE)
        end
    end
end

function style_2d_axes!(ax; xlabel=nothing, ylabel=nothing, title=nothing, labelpad::Real=5)
    ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_SIZE,
                   width=TICK_MAJOR_WIDTH, length=TICK_MAJOR_SIZE, pad=4)
    ax.tick_params(axis="both", which="minor", labelsize=TICK_LABEL_SIZE,
                   width=TICK_MINOR_WIDTH, length=TICK_MINOR_SIZE, pad=4)

    for side in ("left","bottom","right","top")
        ax.spines[side].set_linewidth(AXES_LINEWIDTH)
    end

    if xlabel !== nothing
        ax.set_xlabel(xlabel, labelpad=labelpad)
        set_font!(ax.xaxis.label; size=LABEL_SIZE)
    end
    if ylabel !== nothing
        ax.set_ylabel(ylabel, labelpad=labelpad)
        set_font!(ax.yaxis.label; size=LABEL_SIZE)
    end
    if title !== nothing
        ax.set_title(title, pad=6)
        set_font!(ax.title; size=TITLE_SIZE)
    end

    style_tick_fonts!(ax)
end

function style_3d_axes!(ax; xlabel=nothing, ylabel=nothing, zlabel=nothing, title=nothing, labelpad::Real=1)
    ax.tick_params(axis="x", which="major", labelsize=TICK_LABEL_SIZE,
                   width=TICK_MAJOR_WIDTH, length=TICK_MAJOR_SIZE, pad=2)
    ax.tick_params(axis="y", which="major", labelsize=TICK_LABEL_SIZE,
                   width=TICK_MAJOR_WIDTH, length=TICK_MAJOR_SIZE, pad=2)
    ax.tick_params(axis="z", which="major", labelsize=TICK_LABEL_SIZE,
                   width=TICK_MAJOR_WIDTH, length=TICK_MAJOR_SIZE, pad=2)

    if xlabel !== nothing
        ax.set_xlabel(xlabel, labelpad=0)
        set_font!(ax.xaxis.label; size=LABEL_SIZE - 2)
    end
    if ylabel !== nothing
        ax.set_ylabel(ylabel, labelpad=0)
        set_font!(ax.yaxis.label; size=LABEL_SIZE - 2)
    end
    if zlabel !== nothing
        ax.set_zlabel(zlabel, labelpad=0)
        set_font!(ax.zaxis.label; size=LABEL_SIZE - 2)
    end
    if title !== nothing
        ax.set_title(title, pad=4)
        set_font!(ax.title; size=TITLE_SIZE)
    end

    try
        ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
        ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
        ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    catch
    end
    ax.grid(false)
    style_tick_fonts!(ax; is3d=true)
end

function add_row_label!(fig, y::Real, txt)
    t = fig.text(0.02, y, txt, rotation=90, va="center", ha="center")
    set_font!(t; size=LABEL_SIZE)
end

function panel_positions()
    w = (1 - FIG_L - FIG_R - 2FIG_HG) / 3
    x1 = FIG_L
    x2 = FIG_L + w + FIG_HG
    x3 = FIG_L + 2(w + FIG_HG)

    y4 = FIG_B
    y3 = y4 + ROW4_H + FIG_VG
    y2 = y3 + ROW_H + FIG_VG
    y1 = y2 + ROW_H + FIG_VG

    return (
        smallw = w,
        xs = (x1, x2, x3),
        y1 = y1, y2 = y2, y3 = y3, y4 = y4
    )
end

function title_for_ratio(ratio::Float64)
    if ratio == 0.0
        return L"$\sigma/\sigma_{*}=0$"
    else
        s = @sprintf("%.0f", ratio)
        return latexstring("\$\\sigma/\\sigma_{*}=" * s * "\$")
    end
end

# ============================================================
# Consistent seeding
# ============================================================
single_seed(cfg_seed::Int, ratio::Float64) = cfg_seed + 777_000 + round(Int, 10_000 * ratio)
replicate_seed(cfg_seed::Int, si::Int, r::Int) = cfg_seed + 10_000 * si + r

function sigma_index_for_ratio(sigmas::Vector{Float64}, sigma_star::Float64, ratio::Float64)
    target = ratio * sigma_star
    _, idx = findmin(abs.(sigmas .- target))
    return idx
end

# ============================================================
# Basis: Fourier (tensor-product) and Legendre
# ORTHONORMAL ON THE PHYSICAL INTERVAL [a,b]
# ============================================================
interval_length(a, b) = b - a

function to_theta(X::Matrix{Float64}, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    Θ = zeros(Float64, N, d)
    for j in 1:d
        Θ[:, j] .= (2π .* (X[:, j] .- a[j]) ./ (b[j] - a[j])) .- π
    end
    return Θ
end

"""
1D orthonormal Fourier block on x ∈ [a,b]:
    φ0(x) = 1/sqrt(L)
    φk^c(x) = sqrt(2/L) cos(k θ(x))
    φk^s(x) = sqrt(2/L) sin(k θ(x))
where L = b-a and θ(x) = 2π(x-a)/L - π.
"""
function fourier_1d_block(theta::Vector{Float64}, K::Int, L::Float64)
    N = length(theta)
    B = zeros(Float64, N, 2K + 1)

    B[:, 1] .= 1 / sqrt(L)
    sfac = sqrt(2 / L)

    for k in 1:K
        B[:, 1 + k]     .= sfac .* cos.(k .* theta)
        B[:, 1 + K + k] .= sfac .* sin.(k .* theta)
    end
    return B
end

@inline function fourier_1d_col(kind::Symbol, k::Int, K::Int)
    if kind === :const
        return 1
    elseif kind === :cos
        return 1 + k
    elseif kind === :sin
        return 1 + K + k
    else
        error("Unknown Fourier 1D kind=$(kind).")
    end
end

to_m11(x, a, b) = (2.0 .* (x .- a) ./ (b - a)) .- 1.0

function legendre_polys(z::Vector{Float64}, D::Int)
    N = length(z)
    P = zeros(Float64, N, D + 1)
    P[:, 1] .= 1.0
    if D ≥ 1
        P[:, 2] .= z
    end
    for n in 1:(D - 1)
        P[:, n + 2] .= ((2n + 1) .* z .* P[:, n + 1] .- n .* P[:, n]) ./ (n + 1)
    end
    return P
end

"""
Legendre basis orthonormal on x ∈ [a,b] w.r.t. dx:
    φ_n(x) = sqrt((2n+1)/(b-a)) P_n(z(x)),
    z(x) = 2(x-a)/(b-a) - 1.
"""
function legendre_orthonormal_interval(x::Vector{Float64}, D::Int, a::Float64, b::Float64)
    z = to_m11(x, a, b)
    P = legendre_polys(z, D)
    L = b - a
    for n in 0:D
        P[:, n + 1] .*= sqrt((2n + 1) / L)
    end
    return P
end

function total_degree_multiindex(d::Int, D::Int)
    idx = NTuple{d,Int}[]
    cur = zeros(Int, d)

    function rec!(pos::Int, remaining::Int)
        if pos == d
            cur[pos] = remaining
            push!(idx, Tuple(cur))
            return
        end
        for k in 0:remaining
            cur[pos] = k
            rec!(pos + 1, remaining - k)
        end
    end

    for s in 0:D
        rec!(1, s)
    end
    return idx
end

function build_fourier_design(X::Matrix{Float64}, K::Int, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    Θ = to_theta(X, a, b)

    if d == 1
        L = interval_length(a[1], b[1])
        B = fourier_1d_block(vec(Θ[:, 1]), K, L)
        Φ = B[:, 2:end]  # drop constant; intercept handled separately
        ks = [(k,) for k in 1:K]
        meta = (kind=:fourier, ks=ks, M=K, K=K, L=L, tensor_product=true)
        return Φ, meta
    end

    Φdim = Vector{Matrix{Float64}}(undef, d)
    for j in 1:d
        Lj = interval_length(a[j], b[j])
        Φdim[j] = fourier_1d_block(vec(Θ[:, j]), K, Lj)
    end

    idx = total_degree_multiindex(d, K)
    cols = Vector{Vector{Float64}}()
    terms = Vector{Any}()

    for α in idx
        all(==(0), α) && continue
        active = findall(>(0), α)
        nactive = length(active)

        for trig_choice in Iterators.product(ntuple(_ -> (:cos, :sin), nactive)...)
            col = ones(Float64, N)
            kinds = fill(:const, d)
            @inbounds for (m, j) in enumerate(active)
                k = α[j]
                kind = trig_choice[m]
                kinds[j] = kind
                col .*= @view(Φdim[j][:, fourier_1d_col(kind, k, K)])
            end
            push!(cols, col)
            push!(terms, (alpha=α, kinds=Tuple(kinds)))
        end
    end

    p = length(cols)
    Φ = zeros(Float64, N, p)
    for i in 1:p
        Φ[:, i] .= cols[i]
    end

    meta = (kind=:fourier, idx=idx, terms=terms, K=K, tensor_product=true)
    return Φ, meta
end

function build_legendre_design(X::Matrix{Float64}, D::Int, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    Φdim = Vector{Matrix{Float64}}(undef, d)
    for j in 1:d
        Φdim[j] = legendre_orthonormal_interval(vec(X[:, j]), D, a[j], b[j])
    end

    idx = total_degree_multiindex(d, D)
    p = length(idx)
    Φ = zeros(Float64, N, p)
    for (k, α) in enumerate(idx)
        col = ones(Float64, N)
        @inbounds for j in 1:d
            col .*= @view(Φdim[j][:, α[j] + 1])
        end
        Φ[:, k] = col
    end
    meta = (kind=:legendre, idx=idx)
    return Φ, meta
end

function build_design(cfg, X::Matrix{Float64})
    if cfg.basis == :fourier
        return build_fourier_design(X, cfg.K, cfg.a, cfg.b)
    elseif cfg.basis == :legendre
        return build_legendre_design(X, cfg.D, cfg.a, cfg.b)
    else
        error("Unknown basis=$(cfg.basis). Use :fourier or :legendre.")
    end
end

# ============================================================
# Centering + whitening (CONSISTENT ORDER)
# ============================================================
function invsqrt_gram_centered(Phi::Matrix{Float64}; ridge::Float64)
    N, p = size(Phi)
    μΦ = vec(mean(Phi; dims=1))
    Phi_c = Phi .- reshape(μΦ, 1, :)
    Gc = (Phi_c' * Phi_c) / N
    @inbounds for i in 1:p
        Gc[i, i] += ridge
    end
    F = eigen(Symmetric(Gc))
    λ = F.values
    U = F.vectors
    invsqrtλ = 1.0 ./ sqrt.(max.(λ, 1e-18))
    A = U * Diagonal(invsqrtλ) * U'
    return (A = A, μΦ = μΦ, Gc = Gc, Phi_c = Phi_c)
end

function prepare_centered_whitened_design(Phi0::Matrix{Float64}; use_whitening::Bool, ridge::Float64)
    μΦ = vec(mean(Phi0; dims=1))
    Phi_c = Phi0 .- reshape(μΦ, 1, :)

    if use_whitening
        W = invsqrt_gram_centered(Phi0; ridge=ridge)
        A = W.A
        Phi_fit = Phi_c * A
        return (Phi_fit = Phi_fit, Phi_c = Phi_c, μΦ = μΦ, A = A)
    else
        p = size(Phi0, 2)
        A = Matrix{Float64}(I, p, p)
        Phi_fit = Phi_c
        return (Phi_fit = Phi_fit, Phi_c = Phi_c, μΦ = μΦ, A = A)
    end
end

# ============================================================
# Lasso on centered design (intercept handled separately)
# ============================================================
function fit_lasso_on_centered_design(Phi_fit::Matrix{Float64}, y::Vector{Float64}; lambda::Float64, maxit::Int)
    μy = mean(y)
    y_c = y .- μy

    path = fit(LassoPath, Phi_fit, y_c, Normal();
               α = 1.0, λ = [lambda],
               standardize = false, intercept = false,
               cd_maxiter = maxit)

    B = Lasso.coef(path; select = Lasso.AllSeg())
    c_fit = Vector(B[:, 1])
    return (μy = μy, c_fit = c_fit)
end

function reconstruct_coefficients_and_intercept(
    μy::Float64,
    μΦ::Vector{Float64},
    A::Matrix{Float64},
    c_fit::Vector{Float64};
    use_whitening::Bool
)
    c_raw = use_whitening ? (A * c_fit) : c_fit
    b0 = μy - dot(μΦ, c_raw)
    return (b0 = b0, c_raw = c_raw, c_fit = c_fit)
end

predict_raw(Phi0::Matrix{Float64}, b0::Float64, c_raw::Vector{Float64}) = b0 .+ Phi0 * c_raw

# ============================================================
# Theory helpers
# ============================================================
p_eff_support(c0; eps=1e-8) = max(count(abs.(c0) .> eps), 1)
sigma_star(c0norm, N, p_eff) = c0norm * sqrt(N / max(p_eff, 1e-12))
q_theory(sig::Vector{Float64}, σstar::Float64) = (1 .+ (sig ./ max(σstar, 1e-18)).^2).^(-0.5)

function make_sigmas_log(σmin, σmax, n_sigma; include_sigma0=true)
    npos = include_sigma0 ? (n_sigma - 1) : n_sigma
    @assert npos >= 12
    sig_pos = exp.(range(log(σmin), log(σmax); length=npos))
    return include_sigma0 ? vcat(0.0, sig_pos) : sig_pos
end

# ============================================================
# Stats
# ============================================================
function spectral_entropy_norm(c::Vector{Float64})
    e = abs2.(c)
    s = sum(e)
    s <= 0 && return 0.0
    p = e ./ s
    p2 = clamp.(p, 1e-18, 1.0)
    H = -sum(p2 .* log.(p2))
    return H / log(length(c))
end

function fit_alpha_from_noiseless(c::Vector{Float64}; eps::Float64=1e-14, rmin::Int=3, rmax::Int=0)
    v = sort(abs.(c); rev=true)
    p = length(v)
    rmax = (rmax <= 0) ? p : min(rmax, p)
    ranks = collect(1:rmax)
    sel = (ranks .>= rmin) .& (v[1:rmax] .> eps)
    x = log.(ranks[sel])
    y = log.(v[1:rmax][sel])
    if length(x) < 5
        return 1.0
    end
    slope = cov(x, y) / max(var(x), 1e-18)
    return max(-slope, 1e-6)
end

function fit_powerlaw_reference(c0::Vector{Float64}; eps::Float64=1e-18, rmin::Int=3, rmax::Int=0)
    v = sort(abs.(c0); rev=true)
    p = length(v)
    rmax = (rmax <= 0) ? p : min(rmax, p)
    ranks = collect(1:rmax)

    sel = (ranks .>= rmin) .& (v[1:rmax] .> eps)
    x = log.(ranks[sel])
    y = log.(v[1:rmax][sel])

    if length(x) < 5
        α = 1.0
        C = max(v[1], eps)
        return α, C
    end

    slope = cov(x, y) / max(var(x), 1e-18)
    α = max(-slope, 1e-6)
    a = mean(y) - slope * mean(x)
    C = exp(a)
    return α, C
end

# ============================================================
# Analytic Fourier coefficients for x^2 on [-π, π]
# in the ORIGINAL orthonormal Fourier basis:
#   φ0(x)   = 1/sqrt(2π)
#   φk^c(x) = cos(kx)/sqrt(π), k>=1
#   φk^s(x) = sin(kx)/sqrt(π), k>=1
#
# Then:
#   <x^2, φ0>      = 2π^2 / (3 sqrt(2π))
#   <x^2, φk^c>    = 4 (-1)^k / (sqrt(π) k^2)
#   <x^2, φk^s>    = 0
# ============================================================
function analytic_coeffs_x2_fourier_1d(meta, K::Int)
    @assert meta.kind == :fourier
    @assert hasproperty(meta, :M) "analytic_coeffs_x2_fourier_1d is only for the 1D Fourier case."
    @assert isapprox(meta.L, 2π; atol=1e-12) "analytic_coeffs_x2_fourier_1d assumes interval [-π,π]."

    ks = meta.ks
    M  = meta.M
    c = zeros(Float64, 2M)

    for (i, kvec) in enumerate(ks)
        k = kvec[1]
        if 1 <= k <= K
            c[i] = 4.0 * ((-1.0)^k) / (sqrt(π) * k^2)
        end
    end

    b0 = 2.0 * π^2 / (3.0 * sqrt(2.0 * π))
    return (b0 = b0, c = c)
end

# ============================================================
# Config + Results
# ============================================================
Base.@kwdef mutable struct Config
    seed::Int = 0
    d::Int = 1
    a::Vector{Float64} = [-pi]
    b::Vector{Float64} = [pi]

    n_train::Int = 300
    n_test::Int = 600
    n_rep::Int = 20

    sigma_ratio_min::Float64 = 1e-3
    sigma_ratio_max::Float64 = 20.0
    n_sigma::Int = 240
    include_sigma0::Bool = true
    sigmas::Vector{Float64} = Float64[]

    basis::Symbol = :fourier
    K::Int = 12
    D::Int = 18

    lambda::Float64 = 6e-4
    lasso_maxit::Int = 120_000

    support_eps::Float64 = 1e-8

    use_whitening::Bool = true
    ridge::Float64 = 1e-8
    q_whitened::Bool = true

    coeff_order_1d::Symbol = :sorted_ref

    outdir::String = "out_run"
    save_data::Bool = true
    dpi::Int = 300
end

struct Results
    cfg::Config
    meta::NamedTuple

    Xtr::Matrix{Float64}
    Xte::Matrix{Float64}
    ytr_true::Vector{Float64}
    yte_true::Vector{Float64}

    b0::Float64
    c0::Vector{Float64}
    c0_tilde::Vector{Float64}

    p_eff::Int
    sigma_star::Float64

    rmse_sor::Matrix{Float64}
    q::Matrix{Float64}
    dist_c::Matrix{Float64}
    dist_tilde::Matrix{Float64}

    rmse_med::Vector{Float64}
    q_mean::Vector{Float64}
    q_std::Vector{Float64}
    dist_mean::Vector{Float64}
    dist_tilde_mean::Vector{Float64}

    c_med_per_sigma::Matrix{Float64}
    H_mean_per_sigma::Vector{Float64}
    H_med_per_sigma::Vector{Float64}

    snap_ratios::Vector{Float64}
    snap_sigma_idx::Dict{Float64,Int}
    snap_c_med::Dict{Float64,Vector{Float64}}
    snap_b0_med::Dict{Float64,Float64}
end

# ============================================================
# Core run
# ============================================================
function run(cfg::Config; f_true::Function)
    Random.seed!(cfg.seed)
    ensure_dir(cfg.outdir)

    function sample_X(n)
        X = zeros(Float64, n, cfg.d)
        for j in 1:cfg.d
            X[:, j] .= cfg.a[j] .+ rand(n) .* (cfg.b[j] - cfg.a[j])
        end
        return X
    end

    Xtr = sample_X(cfg.n_train)
    Xte = sample_X(cfg.n_test)

    ytr_true = Float64[f_true(view(Xtr, i, :)) for i in 1:cfg.n_train]
    yte_true = Float64[f_true(view(Xte, i, :)) for i in 1:cfg.n_test]

    Φtr0, meta = build_design(cfg, Xtr)
    Φte0, _    = build_design(cfg, Xte)

    prep = prepare_centered_whitened_design(Φtr0; use_whitening=cfg.use_whitening, ridge=cfg.ridge)
    A   = prep.A
    μΦ  = prep.μΦ
    Φtr = prep.Phi_fit

    fit0_tmp = fit_lasso_on_centered_design(Φtr, ytr_true; lambda=cfg.lambda, maxit=cfg.lasso_maxit)
    rec0 = reconstruct_coefficients_and_intercept(
        fit0_tmp.μy, μΦ, A, fit0_tmp.c_fit; use_whitening=cfg.use_whitening
    )

    b0 = rec0.b0
    c0_tilde = fit0_tmp.c_fit
    c0 = rec0.c_raw

    c0_for_q = cfg.q_whitened ? c0_tilde : c0
    c0norm_q = max(norm(c0_for_q), 1e-18)
    p_eff = p_eff_support(c0_for_q; eps = cfg.support_eps)
    σstar = sigma_star(c0norm_q, cfg.n_train, p_eff)

    σmin = max(cfg.sigma_ratio_min * σstar, 1e-18)
    σmax = max(cfg.sigma_ratio_max * σstar, σmin * 10)
    cfg.sigmas = make_sigmas_log(σmin, σmax, cfg.n_sigma; include_sigma0 = cfg.include_sigma0)

    R = cfg.n_rep
    nσ = length(cfg.sigmas)
    p = length(c0)

    rmse_sor = zeros(R, nσ)
    q = zeros(R, nσ)
    dist_c = zeros(R, nσ)
    dist_tilde = zeros(R, nσ)

    c_med_per_sigma = zeros(Float64, p, nσ)
    H_mean_per_sigma = zeros(Float64, nσ)
    H_med_per_sigma = zeros(Float64, nσ)

    prog = Progress(nσ; desc = "σ-sweep (d=$(cfg.d))", dt = 0.15)
    Htmp = zeros(Float64, R)

    for (si, σ) in enumerate(cfg.sigmas)
        Ctmp = zeros(Float64, R, p)

        for r in 1:R
            Random.seed!(replicate_seed(cfg.seed, si, r))
            ytr = ytr_true .+ σ .* randn(cfg.n_train)

            fit_tmp = fit_lasso_on_centered_design(Φtr, ytr; lambda=cfg.lambda, maxit=cfg.lasso_maxit)
            rec = reconstruct_coefficients_and_intercept(
                fit_tmp.μy, μΦ, A, fit_tmp.c_fit; use_whitening=cfg.use_whitening
            )

            yhat = predict_raw(Φte0, rec.b0, rec.c_raw)
            rmse_sor[r, si] = rmse(yte_true, yhat)

            c_tilde = fit_tmp.c_fit
            c_orig = rec.c_raw
            Ctmp[r, :] .= c_orig

            Htmp[r] = spectral_entropy_norm(c_orig)

            c_for_q = cfg.q_whitened ? c_tilde : c_orig
            q[r, si] = dot(c_for_q, c0_for_q) / (max(norm(c_for_q), 1e-18) * c0norm_q)

            dist_c[r, si] = norm(c_orig - c0)
            dist_tilde[r, si] = norm(c_tilde - c0_tilde)
        end

        c_med_per_sigma[:, si] .= vec(median(Ctmp; dims = 1))
        H_mean_per_sigma[si] = mean(Htmp)
        H_med_per_sigma[si] = median(Htmp)
        next!(prog)
    end

    rmse_med = vec(median(rmse_sor; dims = 1))
    q_mean = vec(mean(q; dims = 1))
    q_std = vec(std(q; dims = 1))
    dist_mean = vec(mean(dist_c; dims = 1))
    dist_tilde_mean = vec(mean(dist_tilde; dims = 1))

    snap_ratios = [1.0, 10.0]
    snap_sigma_idx = Dict{Float64,Int}()
    snap_c_med = Dict{Float64,Vector{Float64}}()
    snap_b0_med = Dict{Float64,Float64}()

    for ratio in snap_ratios
        si = sigma_index_for_ratio(cfg.sigmas, σstar, ratio)
        σsnap = cfg.sigmas[si]
        snap_sigma_idx[ratio] = si

        C = zeros(Float64, R, p)
        B0 = zeros(Float64, R)
        for r in 1:R
            Random.seed!(replicate_seed(cfg.seed, si, r))
            ytr = ytr_true .+ σsnap .* randn(cfg.n_train)

            fit_tmp = fit_lasso_on_centered_design(Φtr, ytr; lambda=cfg.lambda, maxit=cfg.lasso_maxit)
            rec = reconstruct_coefficients_and_intercept(
                fit_tmp.μy, μΦ, A, fit_tmp.c_fit; use_whitening=cfg.use_whitening
            )

            B0[r] = rec.b0
            C[r, :] .= rec.c_raw
        end
        snap_c_med[ratio] = vec(median(C; dims = 1))
        snap_b0_med[ratio] = median(B0)
    end

    res = Results(cfg, meta, Xtr, Xte, ytr_true, yte_true,
                  b0, c0, c0_tilde,
                  p_eff, σstar,
                  rmse_sor, q, dist_c, dist_tilde,
                  rmse_med, q_mean, q_std, dist_mean, dist_tilde_mean,
                  c_med_per_sigma,
                  H_mean_per_sigma, H_med_per_sigma,
                  snap_ratios, snap_sigma_idx, snap_c_med, snap_b0_med)

    if cfg.save_data
        open(joinpath(cfg.outdir, "results.jls"), "w") do io
            serialize(io, res)
        end
    end
    return res
end

# ============================================================
# Helpers for plotting
# ============================================================
function prep_1d_design(res::Results)
    cfg = res.cfg
    xs = range(cfg.a[1], cfg.b[1]; length = 1200)
    Xg = reshape(collect(xs), :, 1)

    Φg0, _  = build_design(cfg, Xg)
    Φtr0, _ = build_design(cfg, res.Xtr)

    prep = prepare_centered_whitened_design(Φtr0; use_whitening=cfg.use_whitening, ridge=cfg.ridge)
    A  = prep.A
    μΦ = prep.μΦ
    Φtr = prep.Phi_fit

    Φg_c = Φg0 .- reshape(μΦ, 1, :)
    Φg   = cfg.use_whitening ? (Φg_c * A) : Φg_c

    return xs, Xg, Φg, Φtr, A, μΦ, Φg0
end

function single_fit_at_ratio(res::Results, ratio::Float64; Φtr=nothing, A=nothing, μΦ=nothing)
    cfg = res.cfg
    σ = ratio * res.sigma_star
    Random.seed!(single_seed(cfg.seed, ratio))
    ytr = res.ytr_true .+ σ .* randn(cfg.n_train)

    fit_tmp = fit_lasso_on_centered_design(Φtr, ytr; lambda=cfg.lambda, maxit=cfg.lasso_maxit)
    rec = reconstruct_coefficients_and_intercept(
        fit_tmp.μy, μΦ, A, fit_tmp.c_fit; use_whitening=cfg.use_whitening
    )
    return (b0 = rec.b0, c_raw = rec.c_raw, c_fit = fit_tmp.c_fit)
end

function median_snapshot_predictor(res::Results, ratio::Float64; A=nothing)
    cfg = res.cfg
    b0m = res.snap_b0_med[ratio]
    cm = res.snap_c_med[ratio]
    if cfg.use_whitening
        ctil = A \ cm
        return (b0 = b0m, c = ctil)
    else
        return (b0 = b0m, c = cm)
    end
end

function get_coeff_estimate(res::Results, ratio::Float64; mode::Symbol=:median)
    cfg = res.cfg
    Φtr0, _ = build_design(cfg, res.Xtr)
    prep = prepare_centered_whitened_design(Φtr0; use_whitening=cfg.use_whitening, ridge=cfg.ridge)
    A = prep.A
    μΦ = prep.μΦ
    Φtr = prep.Phi_fit

    if ratio == 0.0
        return res.c0
    elseif mode == :single
        fit = single_fit_at_ratio(res, ratio; Φtr=Φtr, A=A, μΦ=μΦ)
        return fit.c_raw
    elseif mode == :median
        return res.snap_c_med[ratio]
    else
        error("mode must be :single or :median")
    end
end

# ============================================================
# Panel drawers
# ============================================================
function draw_fit_panel_1d!(ax, res::Results; f_true::Function, ratio::Float64, mode::Symbol=:median)
    cfg = res.cfg
    xs, _, Φg, Φtr, A, μΦ, Φg0 = prep_1d_design(res)
    ytrue = Float64[f_true([x]) for x in xs]

    if ratio == 0.0
        ytr_plot = res.ytr_true
        yhat = predict_raw(Φg0, res.b0, res.c0)
    else
        Random.seed!(single_seed(cfg.seed, ratio))
        σ = ratio * res.sigma_star
        ytr_plot = res.ytr_true .+ σ .* randn(cfg.n_train)

        if mode == :single
            fit_tmp = fit_lasso_on_centered_design(Φtr, ytr_plot; lambda=cfg.lambda, maxit=cfg.lasso_maxit)
            rec = reconstruct_coefficients_and_intercept(
                fit_tmp.μy, μΦ, A, fit_tmp.c_fit; use_whitening=cfg.use_whitening
            )
            yhat = predict_raw(Φg0, rec.b0, rec.c_raw)
        else
            yhat = predict_raw(Φg0, res.snap_b0_med[ratio], res.snap_c_med[ratio])
        end
    end

    ax.scatter(res.Xtr[:, 1], ytr_plot; s = 18, facecolors = "lightgray", edgecolors = "none", zorder = 1)
    ax.plot(xs, ytrue, "k:", lw = 2.6, zorder = 2)
    ax.plot(xs, yhat, "r-", lw = 2.6, zorder = 3)
    ax.set_ylim(-5, 15)
    ax.spines["top"].set_visible(false)
    ax.spines["right"].set_visible(false)
    ax.grid(false)
    style_2d_axes!(ax; xlabel = "Variable x", title = title_for_ratio(ratio))
end

function draw_coeff_panel_1d!(ax, res::Results; ratio::Float64, mode::Symbol=:median,
                              order_mode::Symbol=:sorted_ref)
    cfg = res.cfg
    @assert cfg.basis == :fourier

    c_est = get_coeff_estimate(res, ratio; mode = mode)

    if order_mode == :basis_order
        M = res.meta.M
        c_cos = c_est[1:M]
        ks = [res.meta.ks[i][1] for i in 1:M]
        y_ref = abs.(analytic_coeffs_x2_fourier_1d(res.meta, cfg.K).c[1:M]) .+ 1e-18
        y_est = abs.(c_cos) .+ 1e-18

        ax.plot(ks, y_ref, "k:", lw = 2.6)
        ax.scatter(ks, y_est, c = "r", s = 24)
        ax.set_xlim(0.9, 1.5e1)
        style_2d_axes!(ax; xlabel = "Order k")
    else
        M = res.meta.M
        c0_cos  = res.c0[1:M]
        cest_cos = c_est[1:M]
        
        idx0 = sortperm(abs.(c0_cos); rev = true)
        ranks = collect(1:length(idx0))
        y_ref = abs.(c0_cos[idx0]) .+ 1e-18
        y_est = abs.(cest_cos[idx0]) .+ 1e-18

        ax.plot(ranks, y_ref, "k--", lw = 2.6)
        ax.scatter(ranks, y_est, c = "r", s = 24)
        style_2d_axes!(ax; xlabel = "Noiseless rank")
    end

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.spines["top"].set_visible(false)
    ax.spines["right"].set_visible(false)
    ax.grid(false)
end

function draw_coeff_distribution_1d!(ax, res::Results; ratio::Float64, nbins::Int=40, mode::Symbol=:median)
    c_est = get_coeff_estimate(res, ratio; mode = mode)
    vals = abs.(c_est)
    vals = vals[vals .> 0]

    ax.hist(vals; bins = nbins, density = true, color = "red", alpha = 0.55)
    ax.set_yscale("log")
    ax.set_ylim(1e-3, 10.1)
    ax.set_xlim(-0.2, 6.1)
    ax.set_xticks([0, 2.5, 5])
    ax.spines["top"].set_visible(false)
    ax.spines["right"].set_visible(false)
    ax.grid(false)
    style_2d_axes!(ax; xlabel = L"|c_k|")
end

function draw_entropy_1d!(ax, res::Results; which::Symbol=:median)
    cfg = res.cfg
    sig = copy(cfg.sigmas)
    sig[sig .<= 0] .= minimum(sig[sig .> 0])
    x = sig ./ max(res.sigma_star, 1e-18)

    Hn = which == :median ? res.H_med_per_sigma :
         which == :mean   ? res.H_mean_per_sigma :
         error("which must be :median or :mean")

    p = size(res.c_med_per_sigma, 1)
    c_base = res.c_med_per_sigma[:, 1]
    nnz = max(count(abs.(c_base) .> cfg.support_eps), 1)
    Hn_upper = log(nnz) / log(p)

    n = collect(1:p)
    cpl = 1.0 ./ (n .^ 2)
    Hn_lower = spectral_entropy_norm(cpl)

    ax.plot(x, Hn, "r-", lw = 2.8)
    ax.axhline(Hn_lower; color = "k", lw = 2.6, ls = ":")
    ax.axhline(Hn_upper; color = "k", lw = 2.6, ls = "--")
    ax.set_xscale("log")
    ax.set_xlim(0.9e-2, 1.05 * maximum(x))
    ax.set_ylim(0.0, 1.05)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.spines["top"].set_visible(false)
    ax.spines["right"].set_visible(false)
    ax.grid(false)
    style_2d_axes!(ax; xlabel = L"\sigma/\sigma_{*}")
end

function draw_surface_panel_2d!(ax, res::Results; f_true::Function, ratio::Float64, mode::Symbol=:median)
    cfg = res.cfg
    nx = 120
    x1 = range(cfg.a[1], cfg.b[1]; length = nx)
    x2 = range(cfg.a[2], cfg.b[2]; length = nx)

    X1 = repeat(collect(x1)', nx, 1)
    X2 = repeat(collect(x2), 1, nx)
    Xflat = hcat(vec(X1), vec(X2))

    Φg0, _  = build_design(cfg, Xflat)
    Φtr0, _ = build_design(cfg, res.Xtr)

    prep = prepare_centered_whitened_design(Φtr0; use_whitening=cfg.use_whitening, ridge=cfg.ridge)
    A   = prep.A
    μΦ  = prep.μΦ
    Φtr = prep.Phi_fit

    Φg_c = Φg0 .- reshape(μΦ, 1, :)
    Φg   = cfg.use_whitening ? (Φg_c * A) : Φg_c

    Ztrue = reshape([f_true([u, v]) for v in x2, u in x1], nx, nx)

    if ratio == 0.0
        Zhat = reshape(predict_raw(Φg0, res.b0, res.c0), nx, nx)
        ytr_plot = res.ytr_true
    else
        Random.seed!(single_seed(cfg.seed, ratio))
        σ = ratio * res.sigma_star
        ytr_plot = res.ytr_true .+ σ .* randn(cfg.n_train)

        if mode == :single
            fit_tmp = fit_lasso_on_centered_design(Φtr, ytr_plot; lambda=cfg.lambda, maxit=cfg.lasso_maxit)
            rec = reconstruct_coefficients_and_intercept(
                fit_tmp.μy, μΦ, A, fit_tmp.c_fit; use_whitening=cfg.use_whitening
            )
            Zhat = reshape(predict_raw(Φg0, rec.b0, rec.c_raw), nx, nx)
        else
            Zhat = reshape(predict_raw(Φg0, res.snap_b0_med[ratio], res.snap_c_med[ratio]), nx, nx)
        end
    end

    ax.plot_surface(X1, X2, Zhat;
                    color = "blue",
                    alpha = 0.55,
                    linewidth = 0,
                    antialiased = true,
                    zorder = 1)

    ax.plot_wireframe(X1, X2, Ztrue;
                      color = "black",
                      linewidth = 0.9,
                      alpha = 0.75,
                      rstride = 4,
                      cstride = 4,
                      zorder = 10)

    ax.scatter(res.Xtr[:, 1], res.Xtr[:, 2], ytr_plot;
               s = 12,
               c = "lightgray",
               alpha = 0.70,
               edgecolors = "none",
               depthshade = false,
               zorder = 5)

    ax.set_xlim(cfg.a[1], cfg.b[1])
    ax.set_ylim(cfg.a[2], cfg.b[2])
    ax.set_zlim(-5, 5)

    ax.set_xticks([-1.0, 1.0])
    ax.set_yticks([-1.0, 1.0])
    ax.set_zticks([-5.0, 0.0, 5.0])

    ax.margins(0)
    ax.view_init(elev = 28, azim = -45)

    try
        ax.set_box_aspect((1, 1, 0.75), zoom=1.33)
    catch
    end

    style_3d_axes!(ax; xlabel = "x1", ylabel = "x2", title = title_for_ratio(ratio))
end

function draw_coeff_panel_2d!(ax, res::Results; ratio::Float64, mode::Symbol=:median)
    c_est = get_coeff_estimate(res, ratio; mode = mode)
    idx0 = sortperm(abs.(res.c0); rev = true)
    p = length(idx0)
    ranks = collect(1:p)

    y_est = abs.(c_est[idx0]) .+ 1e-18
    α, C = fit_powerlaw_reference(res.c0; eps = max(res.cfg.support_eps, 1e-18), rmin = 3)
    y_ref = C ./ (ranks .^ α)

    ax.plot(ranks, y_ref, "k--", lw = 2.6, zorder = 1)
    ax.scatter(ranks, y_est, c = "blue", s = 24, zorder = 3)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(0.5e-3, 2.0)
    ax.spines["top"].set_visible(false)
    ax.spines["right"].set_visible(false)
    ax.grid(false)
    style_2d_axes!(ax; xlabel = "Noiseless rank")
end

function draw_coeff_distribution_2d!(ax, res::Results; ratio::Float64, nbins::Int=40, mode::Symbol=:median)
    c_est = get_coeff_estimate(res, ratio; mode = mode)
    vals = abs.(c_est)
    vals = vals[vals .> 0]

    ax.hist(vals; bins = nbins, density = true, color = "blue", alpha = 0.55)
    ax.set_xlim(-0.2, 1.1)
    ax.set_yscale("log")
    ax.set_ylim(1e-3, 7.5e1)
    ax.set_xticks([0, 0.5, 1.0])
    ax.spines["top"].set_visible(false)
    ax.spines["right"].set_visible(false)
    ax.grid(false)
    style_2d_axes!(ax; xlabel = L"|c_k|")
end

function draw_entropy_2d!(ax, res::Results; which::Symbol=:median)
    cfg = res.cfg
    sig = copy(cfg.sigmas)
    sig[sig .<= 0] .= minimum(sig[sig .> 0])
    x = sig ./ max(res.sigma_star, 1e-18)

    Hn = which == :median ? res.H_med_per_sigma :
         which == :mean   ? res.H_mean_per_sigma :
         error("which must be :median or :mean")

    p = size(res.c_med_per_sigma, 1)
    c_base = res.c_med_per_sigma[:, 1]
    nnz = max(count(abs.(c_base) .> cfg.support_eps), 1)
    Hn_upper = log(nnz) / log(p)

    n = collect(1:p)
    α = fit_alpha_from_noiseless(c_base; eps = cfg.support_eps, rmin = 3)
    cpl = 1.0 ./ (n .^ α)
    Hn_lower = spectral_entropy_norm(cpl)

    ax.plot(x, Hn, color = "blue", lw = 2.8)
    ax.axhline(Hn_lower; color = "k", lw = 2.6, ls = ":")
    ax.axhline(Hn_upper; color = "k", lw = 2.6, ls = "--")
    ax.set_xscale("log")
    ax.set_xlim(0.9e-2, 1.05 * maximum(x))
    ax.set_ylim(0.0, 1.05)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.spines["top"].set_visible(false)
    ax.spines["right"].set_visible(false)
    ax.grid(false)
    style_2d_axes!(ax; xlabel = L"\sigma/\sigma_{*}")
end

# ============================================================
# Combined figure builders
# ============================================================
function plot_combined_1d(res::Results; f_true::Function, ratios::Vector{Float64}=[0.0, 1.0, 10.0],
                          mode::Symbol=:median, which_entropy::Symbol=:median,
                          outname_noext::String="figs_comb_1D")
    ensure_dir(res.cfg.outdir)
    pos = panel_positions()

    fig = figure(figsize = (12.8, 13.2))

    axs_r1 = [fig.add_axes([pos.xs[j], pos.y1, pos.smallw, ROW_H]) for j in 1:3]
    axs_r2 = [fig.add_axes([pos.xs[j], pos.y2, pos.smallw, ROW_H]) for j in 1:3]
    axs_r3 = [fig.add_axes([pos.xs[j], pos.y3, pos.smallw, ROW_H]) for j in 1:3]
    ax_r4 = fig.add_axes([FIG_L, pos.y4, 1 - FIG_L - FIG_R, ROW4_H])

    for (j, ratio) in enumerate(ratios)
        draw_fit_panel_1d!(axs_r1[j], res; f_true = f_true, ratio = ratio, mode = mode)
        draw_coeff_panel_1d!(axs_r2[j], res; ratio = ratio, mode = mode, order_mode = res.cfg.coeff_order_1d)
        draw_coeff_distribution_1d!(axs_r3[j], res; ratio = ratio, mode = mode)
    end
    draw_entropy_1d!(ax_r4, res; which = which_entropy)

    add_row_label!(fig, pos.y1 + ROW_H / 2, "Function f(x)")
    add_row_label!(fig, pos.y2 + ROW_H / 2, L"c_k")
    add_row_label!(fig, pos.y3 + ROW_H / 2, L"P(|c_k|)")
    add_row_label!(fig, pos.y4 + ROW4_H / 2, "Norm. entropy")

    save_pdf(joinpath(res.cfg.outdir, outname_noext); dpi = res.cfg.dpi)
    close(fig)
end

function plot_combined_2d(res::Results; f_true::Function, ratios::Vector{Float64}=[0.0, 1.0, 10.0],
                          mode::Symbol=:median, which_entropy::Symbol=:median,
                          outname_noext::String="figs_comb_2D")
    ensure_dir(res.cfg.outdir)
    pos = panel_positions()

    fig = figure(figsize = (12.8, 13.2))

    axs_r1 = [fig.add_axes([pos.xs[j], pos.y1, pos.smallw, ROW_H], projection = "3d") for j in 1:3]
    axs_r2 = [fig.add_axes([pos.xs[j], pos.y2, pos.smallw, ROW_H]) for j in 1:3]
    axs_r3 = [fig.add_axes([pos.xs[j], pos.y3, pos.smallw, ROW_H]) for j in 1:3]
    ax_r4 = fig.add_axes([FIG_L, pos.y4, 1 - FIG_L - FIG_R, ROW4_H])

    for (j, ratio) in enumerate(ratios)
        draw_surface_panel_2d!(axs_r1[j], res; f_true = f_true, ratio = ratio, mode = mode)
        draw_coeff_panel_2d!(axs_r2[j], res; ratio = ratio, mode = mode)
        draw_coeff_distribution_2d!(axs_r3[j], res; ratio = ratio, mode = mode)
    end
    draw_entropy_2d!(ax_r4, res; which = which_entropy)

    add_row_label!(fig, pos.y1 + ROW_H / 2, "Function f(x)")
    add_row_label!(fig, pos.y2 + ROW_H / 2, L"c_k")
    add_row_label!(fig, pos.y3 + ROW_H / 2, L"P(|c_k|)")
    add_row_label!(fig, pos.y4 + ROW4_H / 2, "Norm. entropy")

    save_pdf(joinpath(res.cfg.outdir, outname_noext); dpi = res.cfg.dpi)
    close(fig)
end

# ============================================================
# Run configs
# ============================================================
f_x2(x) = x[1]^2
f_2d(x) = sin(x[1]^2 * exp(x[2]))

function run_1d()
    cfg = Config(
        seed = 0, d = 1, a = [-pi], b = [pi],
        n_train = 300, n_test = 600, n_rep = 160,
        sigma_ratio_min = 1e-3, sigma_ratio_max = 20.0, n_sigma = 240, include_sigma0 = true,
        basis = :fourier, K = 15,
        lambda = 6e-4, lasso_maxit = 140_000,
        use_whitening = true, ridge = 1e-8, q_whitened = true,
        coeff_order_1d = :sorted_ref,
        outdir = "figs",
        save_data = true,
        dpi = 300
    )
    return run(cfg; f_true = f_x2)
end

function run_2d()
    cfg = Config(
        seed = 1, d = 2, a = [-1.0, -1.0], b = [1.0, 1.0],
        n_train = 1200, n_test = 2400, n_rep = 80,
        sigma_ratio_min = 1e-3, sigma_ratio_max = 20.0, n_sigma = 240, include_sigma0 = true,
        basis = :legendre, D = 18,
        lambda = 6e-4, lasso_maxit = 160_000,
        use_whitening = true, ridge = 1e-8, q_whitened = true,
        outdir = "figs",
        save_data = true,
        dpi = 300
    )
    return run(cfg; f_true = f_2d)
end

# ============================================================
# MAIN
# ============================================================
RESULTS_1D = run_1d()
RESULTS_2D = run_2d()

plot_combined_1d(RESULTS_1D; f_true = f_x2, mode = :median, which_entropy = :median,
                 outname_noext = "figs_comb_1D")
plot_combined_2d(RESULTS_2D; f_true = f_2d, mode = :median, which_entropy = :median,
                 outname_noext = "figs_comb_2D")

println("\nDone.")
println("1D combined PDF saved in: $(joinpath(RESULTS_1D.cfg.outdir, "figs_comb_1D.pdf"))")
println("2D combined PDF saved in: $(joinpath(RESULTS_2D.cfg.outdir, "figs_comb_2D.pdf"))")
println("1D coefficient order mode available: :sorted_ref (default) or :basis_order")

σ-sweep (d=1) 100%|██████████████████████████████████████| Time: 0:00:07
σ-sweep (d=2) 100%|██████████████████████████████████████| Time: 0:01:59



Done.
1D combined PDF saved in: figs\figs_comb_1D.pdf
2D combined PDF saved in: figs\figs_comb_2D.pdf
1D coefficient order mode available: :sorted_ref (default) or :basis_order
